In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning_scripts import lightning_ssl
sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path

import lightning_scripts.eval_jsin_transfer_matched as transfer

/mnt/ceph/users/igriffith/projects/cochdnn/byol-a/byol_a/common.py:31: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")


In [2]:
torch.cuda.is_available()

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


False

In [3]:
## init config. Will be yaml eventually, but start as dict 
config_path = Path("model_configs/resnet18_barlow_equivariant_lmbda_1e-2_lr_2e-1_w_invar_augment_no_avgpool_eq_lmbda_1e-02.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

# Overwrite config for linear classifier 
config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['hparas']['optimizer'] = "AdamW"
config['hparas']['lr'] = 0.0001
config['data']['eval_max'] = 1
config['model']['arch_kwargs']['supervised'] = False
config['model']['arch_kwargs']['num_classes'] = {"signal/word_int": 794,    
                            "signal/speaker_int": 433} 

config['hparas']['task_loss_params'] = {key:value for key,value in config['hparas']['task_loss_params'].items() if key in config['model']['arch_kwargs']['num_classes'].keys()}
## Try with MLP classifier 


# config['model']['task'] = {"signal/word_int": 794,    
#                             "signal/speaker_int": 433}  
config['model']['arch_kwargs']['time_average'] = True  

In [4]:
config['model']['arch_kwargs'].get('time_average')

True

In [5]:
model_ckpt_dir = "model_checkpoints"
checkpoint_dir = Path(model_ckpt_dir) / f"{config_path.stem}/checkpoints"
ckpt_paths = sorted(checkpoint_dir.glob("*.ckpt"), key=os.path.getctime)
ckpt_path = ckpt_paths[-1] # get latest checkpoint 
print(ckpt_path)

model_checkpoints/resnet18_barlow_equivariant_lmbda_1e-2_lr_2e-1_w_invar_augment_no_avgpool_eq_lmbda_1e-02/checkpoints/epoch=105-step=23850-best_train.ckpt


In [6]:
importlib.reload(transfer)
SSLWordClassifier = transfer.SSLClassifier

module = SSLWordClassifier(config=config,
                           ckpt_path=ckpt_path,
                           layer_out='layer4')

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['model.model.lin_cls.signal/word_int.weight', 'model.model.lin_cls.signal/word_int.bias', 'model.model.lin_cls.signal/speaker_int.weight', 'model.model.lin_cls.signal/speaker_int.bias', 'model.model.lin_cls.noise/labels_int.weight', 'model.model.lin_cls.noise/labels_int.bias']


In [12]:
from jsinV3DataLoader_precombined_batched import jsinV3_precombined_all_signals

dataset = jsinV3_precombined_all_signals(root="/mnt/ceph/users/jfeather/data/training_datasets_audio/JSIN_all_v3/subsets/",
                                                 train=True,
                                                 transform=None, # perform transforms in collate_fn
                                                 batch_size=10)

In [31]:
fg, bg, labels = dataset[0]
labels[task] = torch.from_numpy(labels[task])
fg = torch.from_numpy(fg)


In [34]:
task = 'signal/word_int'
task_egs = (labels[task] != 0 ).nonzero(as_tuple=True)

bg[task_egs].shape

(2, 40000)

In [11]:

trainer = L.Trainer(devices=1)
trainer.fit(module)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name              | Type                   | Params | Mode 
---------------------------------------------------------------------
0 | feature_extractor | OptimizedModule        | 59.4 M | eval 
1 | transforms        | AudioCompose           | 0      | train
2 | classifier        | ModuleDict             | 4.4 M  | train
3 | multi_task_loss   | jsinV3_multi_task_loss | 0      | train
4 | accuracy          | ModuleDict             | 0      | train
---------------------------------------------------------------------
4.4 M     Trainable params
59.4 M    Non-trainable params
63.8 M    Total params
255.222   Total estimated model params size (MB)
8         Modules in train mode
98        Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

torch.Size([64, 512, 7, 13])
torch.Size([64, 512, 7, 13])



Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [9]:
outputs = trainer.predict(module, module.val_dataloader(), return_predictions=True)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [14]:
top1_word = []
top1_speaker = []
top5_word = []
top5_speaker = []

for record in outputs:
    top1_word.append(record['top1']['signal/word_int'])
    top1_speaker.append(record['top1']['signal/speaker_int'])
    top5_word.append(record['top5']['signal/word_int'])
    top5_speaker.append(record['top5']['signal/speaker_int'])

In [30]:
n_examples = len(outputs)
output_dict = {
    "word_top1_mean": torch.stack(top1_word).mean(),
    "word_top1_sem": torch.stack(top1_word).std() / np.sqrt(n_examples),
    "speaker_top1_mean": torch.stack(top1_speaker).mean(),
    "speaker_top1_sem": torch.stack(top1_speaker).std() / np.sqrt(n_examples),

    "word_top5_mean": torch.stack(top5_word).mean(),
    "word_top5_sem": torch.stack(top5_word).std() / np.sqrt(n_examples),
    "speaker_top5_mean": torch.stack(top5_speaker).mean(),
    "speaker_top5_sem": torch.stack(top5_speaker).std() / np.sqrt(n_examples),
}
output_dict = {key:val.item() for key,val in output_dict.items()}

In [31]:
output_dict

{'word_top1_mean': 0.0341125950217247,
 'word_top1_sem': 0.0013719532871618867,
 'speaker_top1_mean': 0.24576574563980103,
 'speaker_top1_sem': 0.0032539963722229004,
 'word_top5_mean': 0.5456822514533997,
 'word_top5_sem': 0.00541748246178031,
 'speaker_top5_mean': 0.7976502776145935,
 'speaker_top5_sem': 0.0033717849291861057}

In [12]:
output_dict = {}
for key, task_dict in outputs.items():
    for task, scores in task_dict.items():
        n_examples = len(scores)
        task_str = task.split('/')[-1]
        output_dict[f"{task_str}_{key}_mean"] = torch.cat(scores).mean()
        output_dict[f"{task_str}_{key}_sem"] = torch.cat(scores).std() / np.sqrt(n_examples)

AttributeError: 'list' object has no attribute 'items'

In [11]:
output_vals = torch.cat([output['accuracy'] for output in outputs])
len(output_vals)

KeyError: 'accuracy'

In [ ]:
output_vals.mean()

tensor(0.0010)

In [ ]:

output_vals.std(unbiased=True) / (output_vals.size(0) ** 0.5)

tensor(0.0002)

In [ ]:
import pickle
with open('eval_jsin_results/ssl_barlow_word_resnet50_hparam_set_0_linear_eval_jsin.pkl', 'rb') as handle:

    results = pickle.load(handle)

In [ ]:
results

{'mean_acc': tensor(0.0018),
 'std_acc': tensor(0.0423),
 'sem_acc': tensor(0.0003)}